# 04 - RUL Prediction

This notebook trains and evaluates RUL models with the same cross-battery split and visualizes cycle-wise error behavior.

In [1]:
from pathlib import Path
import sys
import random
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'
METRICS_PATH = PROJECT_ROOT / 'results' / 'metrics.csv'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## Load feature dataset

In [2]:
feature_df = pd.read_csv(PROCESSED_DIR / 'features_all.csv')
feature_df.head()

,battery_id,cycle_number,capacity_Ah,avg_voltage,avg_current,avg_temperature,charge_time,discharge_time,energy_charged_Wh,SOH,...,max_temperature,temperature_rise,discharge_duration,energy_discharged,internal_resistance_proxy,capacity_fade_rate,cycle_efficiency,capacity_lag_1,capacity_lag_3,capacity_lag_5
0,B0005,1,1.862203,3.529829,1.818712,32.572328,7597.875,3690.234,3.276268,93.110144,...,38.982181,14.652147,3690.234,6.608778,0.216948,-0.004175,2.017167,1.862203,1.862203,1.862203
1,B0005,2,1.852078,3.537320,1.817644,32.725235,10516.000,3672.344,7.638814,92.603889,...,39.033398,14.335646,3672.344,6.586345,0.990131,-0.004175,0.862221,1.862203,1.862203,1.862203
2,B0005,3,1.841049,3.543737,1.816542,32.642862,10484.547,3651.641,7.608577,92.052437,...,38.818797,14.084531,3651.641,6.555683,26.274884,-0.004175,0.861618,1.852078,1.862203,1.862203
3,B0005,4,1.840912,3.543666,1.825618,32.514876,10397.890,3631.563,7.578063,92.045600,...,38.762305,14.108068,3631.563,6.554829,0.235616,-0.004175,0.864974,1.841049,1.862203,1.862203
4,B0005,5,1.840360,3.542343,1.826148,32.382349,10495.203,3629.172,7.565826,92.017996,...,38.665393,14.140596,3629.172,6.552232,0.094378,-0.004175,0.866030,1.840912,1.852078,1.862203


## Train RUL models

In [3]:
from src.models import run_rul_benchmark

rul_metrics, rul_results = run_rul_benchmark(
    feature_df=feature_df,
    models_dir=MODELS_DIR,
    test_battery='B0018',
    sequence_length=10,
    include_transformer=True,
)
rul_metrics_df = pd.DataFrame(rul_metrics).sort_values('RMSE').reset_index(drop=True)
rul_metrics_df

,Model,Task,RMSE,MAE,MAPE,R2,Train_Time_s
0,Random Forest Regressor,RUL,6.373208,5.091485,17.973037,0.962434,0.270038
1,XGBoost Regressor,RUL,6.385948,4.935395,15.623504,0.962284,4.146312
2,Transformer Encoder,RUL,9.119315,7.989763,66.475209,0.907624,5.507179
3,LSTM Neural Network,RUL,33.644478,29.553264,200.059597,-0.257366,1.096300


## Plot actual/predicted RUL and cycle-wise error

In [4]:
import numpy as np
from src.visualisation import plot_actual_vs_predicted, plot_rul_error

for model_name, payload in rul_results.items():
    safe_name = model_name.lower().replace(' ', '_').replace('-', '_')
    plot_actual_vs_predicted(
        cycle_number=np.asarray(payload['cycle_number']),
        y_true=np.asarray(payload['y_true']),
        y_pred=np.asarray(payload['predictions']),
        title=f'RUL: Actual vs Predicted ({model_name}) on B0018',
        y_label='RUL (cycles)',
        output_path=FIG_DIR / f'rul_actual_vs_pred_{safe_name}.png',
    )
    plot_rul_error(
        cycle_number=np.asarray(payload['cycle_number']),
        y_true=np.asarray(payload['y_true']),
        y_pred=np.asarray(payload['predictions']),
        output_path=FIG_DIR / f'rul_error_{safe_name}.png',
    )

print('Saved RUL prediction and error plots to results/figures/.')

Saved RUL prediction and error plots to results/figures/.
